# 02- Construção das tabelas dimensões
---
## Objetivo

Este notebook tem como objetivo construir as tabelas dimensão utilizadas no modelo dimensional do projeto de Business Intelligence sobre assistência estudantil da UFPB.

As dimensões são obtidas a partir das consultas realizadas no SEDAP+ e passam por etapas de verificação, tratamento e padronização antes de serem utilizadas na construção da tabela fato e do dashboard.

 ---

## Importação das bibliotecas necessárias

In [2]:
import pandas as pd
from pathlib import Path
import sys

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

In [ ]:
from src.dimensions.dim_curso import criar_dim_curso
from src.dimensions.dim_curso_campus1 import criar_dim_curso_campus1

from src.dimensions.dim_sexo import criar_dim_sexo
from src.dimensions.dim_raca import criar_dim_raca
from src.dimensions.dim_turno import criar_dim_turno
from src.dimensions.dim_grau import criar_dim_grau
from src.dimensions.dim_modalidade import criar_dim_modalidade

from src.dimensions.dim_centro import criar_dim_centro
from src.dimensions.dim_auxilio import criar_dim_auxilio

 ---
# 1. Construção da Dimensão Curso
## 1.1 Carregar os dados

In [4]:
df_curso = pd.read_csv(
    ROOT / "data/raw/SEDAP/alunos_por_curso.csv"
)

## 1.2 Criar Dimensão

In [5]:
dim_curso = criar_dim_curso(df_curso)

In [6]:
#Validar
display(dim_curso.head())

dim_curso.info()

dim_curso.isna().sum()

dim_curso["ID_CURSO"].duplicated().sum()

,ID_CURSO,CURSO,CO_CINE_ROTULO,GRAU_ACADEMICO,MODALIDADE_ENSINO,TOTAL_ALUNOS
0,13397,CIÊNCIAS CONTÁBEIS,0411C01,Bacharelado,Presencial,1438
1,13418,PEDAGOGIA,0113P01,Licenciatura,Presencial,1382
2,13398,DIREITO,0421D01,Bacharelado,Presencial,1137
3,13424,MEDICINA,0912M01,Bacharelado,Presencial,844
4,107548,LETRAS - LÍNGUA PORTU,0115L13,Licenciatura,Presencial,766


<class 'pandas.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ID_CURSO           119 non-null    int64
 1   CURSO              119 non-null    str  
 2   CO_CINE_ROTULO     119 non-null    str  
 3   GRAU_ACADEMICO     119 non-null    str  
 4   MODALIDADE_ENSINO  119 non-null    str  
 5   TOTAL_ALUNOS       119 non-null    int64
dtypes: int64(2), str(4)
memory usage: 10.7 KB


np.int64(0)

In [7]:
#Salvar
dim_curso.to_csv(
    ROOT / "data/processed/Dimensões/dim_curso.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Construção da Dimensão Curso (filtro Campus I)



In [8]:
dim_curso_campus1 = criar_dim_curso_campus1(dim_curso)

In [9]:
# Validar
display(dim_curso_campus1.head())

print(len(dim_curso_campus1))

,ID_CURSO,CURSO,CO_CINE_ROTULO,GRAU_ACADEMICO,MODALIDADE_ENSINO,TOTAL_ALUNOS
0,13394,CIÊNCIAS ECONÔMICAS,0311E01,Bacharelado,Presencial,473
1,13395,ADMINISTRAÇÃO,0413A01,Bacharelado,Presencial,761
2,13396,BIBLIOTECONOMIA,0322B01,Bacharelado,Presencial,562
3,13397,CIÊNCIAS CONTÁBEIS,0411C01,Bacharelado,Presencial,1438
4,13398,DIREITO,0421D01,Bacharelado,Presencial,1137


93


In [10]:
# Salvar
dim_curso_campus1.to_csv(
    ROOT / "data/processed/Dimensões/dim_curso_campus1.csv",
    index=False,
    encoding="utf-8-sig"
)

## 3. Dimensão Sexo

In [11]:
#Leitura
df_sexo = pd.read_csv(
    ROOT / "data/raw/SEDAP/sexo_sedap.csv"
)

In [15]:
#Criar dimensão
dim_sexo = criar_dim_sexo(df_sexo)

In [16]:
#Validação
display(dim_sexo)

dim_sexo.info()

dim_sexo.isna().sum()

dim_sexo.duplicated().sum()

,ID_SEXO,TOTAL_ALUNOS,DESCRICAO
0,1,21244,Masculino
1,2,18906,Feminino


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID_SEXO       2 non-null      int64
 1   TOTAL_ALUNOS  2 non-null      int64
 2   DESCRICAO     2 non-null      str  
dtypes: int64(2), str(1)
memory usage: 198.0 bytes


np.int64(0)

In [ ]:
#Salvar
dim_sexo.to_csv(
    ROOT / "data/processed/Dimensões/dim_sexo.csv",
    index=False,
    encoding="utf-8-sig"
)

## 4. Dimensão Raça

In [ ]:
df_raca = pd.read_csv(
    ROOT / "data/raw/SEDAP/raca_sedap.csv"
)

In [ ]:
dim_raca = criar_dim_raca(df_raca)

In [ ]:
display(dim_raca)

dim_raca.info()

dim_raca.isna().sum()

dim_raca.duplicated().sum()

dim_raca["ID_RACA"].duplicated().sum()

In [ ]:
assert set(dim_raca["ID_RACA"]) == {0, 1, 2, 3, 4, 5}

In [ ]:
dim_raca.to_csv(
    ROOT / "data/processed/Dimensões/dim_raca.csv",
    index=False,
    encoding="utf-8-sig"
)

## 5. Dimensão Turno (adição de "Não informado")

Existem cursos **EaD sem turno preenchido** no SEDAP+. Para não descartar
esses registros na fato, criamos aqui o registro `ID_TURNO = 0` ->
`"Não informado"`.

Na tabela fato, os valores nulos de turno serão substituídos por
`ID_TURNO = 0` (ver notebook da Fato, etapa "Atualizar a dimensão turno").

In [ ]:
df_turno = pd.read_csv(
    ROOT / "data/raw/SEDAP/turno_sedap.csv"
)

In [ ]:
dim_turno = criar_dim_turno(df_turno)

In [ ]:
display(dim_turno)

dim_turno.info()

dim_turno.isna().sum()

dim_turno.duplicated().sum()

dim_turno["ID_TURNO"].duplicated().sum()

In [ ]:
assert set(dim_turno["ID_TURNO"]) == {0, 1, 2, 3, 4}

In [ ]:
dim_turno.to_csv(
    ROOT / "data/processed/Dimensões/dim_turno.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
colunas_esperadas = {"ID_TURNO", "TOTAL_ALUNOS", "DESCRICAO"}

if not colunas_esperadas.issubset(dim_turno.columns):
    raise ValueError("A base de turno não possui todas as colunas esperadas.")

## 6. Dimensão Grau

Derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [ ]:
dim_grau = criar_dim_grau(dim_curso)

In [ ]:
display(dim_grau)

dim_grau.info()

dim_grau.isna().sum()

dim_grau.duplicated().sum()

In [ ]:
assert set(dim_grau["GRAU_ACADEMICO"]) == {
    "Bacharelado",
    "Licenciatura",
    "Tecnológico"
}

In [ ]:
dim_grau.to_csv(
    ROOT / "data/processed/Dimensões/dim_grau.csv",
    index=False,
    encoding="utf-8-sig"
)

## 7. Dimensão Modalidade

Também derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [ ]:
dim_modalidade = criar_dim_modalidade(dim_curso)

In [ ]:
display(dim_modalidade)

dim_modalidade.info()

dim_modalidade.isna().sum()

dim_modalidade.duplicated().sum()

In [ ]:
assert set(dim_modalidade["MODALIDADE_ENSINO"]) == {
    "EaD",
    "Presencial"
}

In [ ]:
dim_modalidade.to_csv(
    ROOT / "data/processed/Dimensões/dim_modalidade.csv",
    index=False,
    encoding="utf-8-sig"
)

## Construção da dimensão auxilio

In [ ]:
dim_auxilio = criar_dim_auxilio()

In [ ]:
display(dim_auxilio)

dim_auxilio.info()

dim_auxilio.isna().sum()

dim_auxilio.duplicated().sum()

In [ ]:
assert len(dim_auxilio) == 6
assert dim_auxilio["ID_AUXILIO"].is_unique

In [ ]:
dim_auxilio.to_csv(
    ROOT / "data/processed/Dimensões/dim_auxilio.csv",
    index=False,
    encoding="utf-8-sig"
)

## Criação da dimensão Centro

In [ ]:
dim_centro = criar_dim_centro()

In [ ]:
display(dim_centro)

dim_centro.info()

dim_centro.isna().sum()

dim_centro.duplicated().sum()

In [ ]:
assert len(dim_centro) == 13
assert dim_centro["ID_CENTRO"].is_unique

In [ ]:
dim_centro.to_csv(
    ROOT / "data/processed/Dimensões/dim_centro.csv",
    index=False,
    encoding="utf-8-sig"
)